In [0]:
%pip install google-cloud-bigquery
%pip install azure-storage-blob
%pip install db-dtypes

%pip install google-cloud-bigquery-storage
%pip install pandas-gbq
%pip install pyarrow

In [0]:
# =========================
# Inicialização Databricks
# =========================

# ------------------------------------------
# Detecta ambiente Databricks
# ------------------------------------------

try:

    # Verifica se dbutils está disponível

    dbutils.notebook.entry_point

    IS_DATABRICKS = True

    print(
        "✅ Ambiente Databricks detectado."
    )

except NameError:

    IS_DATABRICKS = False

    print(
        "⚠️ dbutils não encontrado. "
        "Executando em ambiente local."
    )

# ------------------------------------------
# Informações ambiente
# ------------------------------------------

print(
    f"✅ Databricks: "
    f"{IS_DATABRICKS}"
)

# ------------------------------------------
# Caminho atual
# ------------------------------------------

from pathlib import Path

print(
    f"✅ Diretório atual: "
    f"{Path.cwd()}"
)

# ------------------------------------------
# Conclusão
# ------------------------------------------

print(
    "✅ Inicialização concluída."
)

In [0]:
# =========================
# Imports
# =========================

from pathlib import Path
import os
import tempfile

import db_dtypes

# =========================
# Imports Projeto
# =========================

import sys

# sobe um nível a partir de jobs/

if str(Path.cwd().parent) not in sys.path:
    sys.path.append(str(Path.cwd().parent))

from src.config.secrets import get_secret

# =========================
# Definir raiz do projeto
# =========================

BASE_DIR = Path.cwd()

while not (BASE_DIR / "src").exists():

    if BASE_DIR.parent == BASE_DIR:

        raise FileNotFoundError(
            "❌ Pasta 'src' não encontrada em nenhum nível acima."
        )

    BASE_DIR = BASE_DIR.parent

print(
    f"✅ BASE_DIR localizado: "
    f"{BASE_DIR}"
)

# =========================
# Azure Storage
# =========================

AZURE_STORAGE_ACCOUNT = get_secret(
    "AZURE-STORAGE-ACCOUNT",
    "AZURE_STORAGE_ACCOUNT"
)

AZURE_STORAGE_KEY = get_secret(
    "AZURE-STORAGE-KEY",
    "AZURE_STORAGE_KEY"
)

# =========================
# Google BigQuery
# =========================

GOOGLE_CREDENTIALS_JSON = get_secret(
    "GOOGLE-APPLICATION-CREDENTIALS-JSON",
    "GOOGLE_APPLICATION_CREDENTIALS_JSON"
)

cred_path = None

# =========================
# Credenciais GCP
# =========================

if GOOGLE_CREDENTIALS_JSON:

    temp_file = tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".json",
        delete=False,
        encoding="utf-8"
    )

    temp_file.write(
        GOOGLE_CREDENTIALS_JSON
    )

    temp_file.close()

    cred_path = temp_file.name

    print(
        "✅ Credenciais BigQuery carregadas do Key Vault."
    )

else:

    cred_file = (
        "tough-medley-505300-k1-164371097431.json"
    )

    local_cred_path = (
        BASE_DIR
        / "credenciais"
        / cred_file
    )

    if not local_cred_path.exists():

        raise FileNotFoundError(
            f"❌ Arquivo não encontrado: "
            f"{local_cred_path}"
        )

    cred_path = str(
        local_cred_path
    )

    print(
        f"✅ Credenciais locais carregadas: "
        f"{cred_path}"
    )

# =========================
# Variáveis de Ambiente
# =========================

os.environ[
    "GOOGLE_APPLICATION_CREDENTIALS"
] = str(
    cred_path
)

if AZURE_STORAGE_ACCOUNT:

    os.environ[
        "AZURE_STORAGE_ACCOUNT"
    ] = AZURE_STORAGE_ACCOUNT

if AZURE_STORAGE_KEY:

    os.environ[
        "AZURE_STORAGE_KEY"
    ] = AZURE_STORAGE_KEY

print(
    "✅ Variáveis de ambiente configuradas."
)

# =========================
# Validações
# =========================

if not Path(
    cred_path
).exists():

    raise FileNotFoundError(
        f"❌ Arquivo de credenciais não encontrado: "
        f"{cred_path}"
    )

if not AZURE_STORAGE_ACCOUNT:

    raise ValueError(
        "❌ AZURE-STORAGE-ACCOUNT não configurada."
    )

if not AZURE_STORAGE_KEY:

    raise ValueError(
        "❌ AZURE-STORAGE-KEY não configurada."
    )

# =========================
# Logs
# =========================

print(
    f"✅ Arquivo credencial: "
    f"{cred_path}"
)

print(
    f"✅ Storage Account: "
    f"{AZURE_STORAGE_ACCOUNT}"
)

print(
    f"✅ Storage Key carregada: "
    f"{bool(AZURE_STORAGE_KEY)}"
)

print(
    "✅ Inicialização concluída."
)

In [0]:
# =========================
# Imports Projeto
# =========================

import sys
from pathlib import Path

# sobe um nível a partir de jobs/

if str(Path.cwd().parent) not in sys.path:
    sys.path.append(str(Path.cwd().parent))

from src.config.secrets import get_secret

# =========================
# Container Bronze
# =========================

BRONZE_CONTAINER = get_secret(
    "AZURE-CONTAINER-BRONZE",
    "AZURE_CONTAINER_BRONZE"
)

if not BRONZE_CONTAINER:
    raise ValueError(
        "❌ Container Bronze não configurado."
    )

# =========================
# Tabelas Databricks
# =========================

TABLES = []

try:

    # ------------------------------------------
    # Widget para execução via Job
    # ------------------------------------------

    dbutils.widgets.text(
        "TABLES",
        ""
    )

    tables_widget = (
        dbutils.widgets.get(
            "TABLES"
        )
    )

    if tables_widget:

        TABLES = [
            table.strip()
            for table in tables_widget.split(",")
            if table.strip()
        ]

    print(
        "✅ Widgets Databricks carregados."
    )

except NameError:

    print(
        "⚠️ dbutils não encontrado. "
        "Executando localmente."
    )

# =========================
# Validações
# =========================

if not AZURE_STORAGE_ACCOUNT:
    raise ValueError(
        "❌ AZURE_STORAGE_ACCOUNT não configurada."
    )

if not AZURE_STORAGE_KEY:
    raise ValueError(
        "❌ AZURE_STORAGE_KEY não configurada."
    )

# =========================
# Logs
# =========================

print(
    f"✅ Container Bronze: "
    f"{BRONZE_CONTAINER}"
)

print(
    f"✅ Quantidade de tabelas: "
    f"{len(TABLES)}"
)

if TABLES:

    print(
        "✅ Tabelas configuradas:"
    )

    for table in TABLES:

        print(
            f"   • {table}"
        )

else:

    print(
        "⚠️ Nenhuma tabela informada."
    )

print(
    f"✅ Storage Account: "
    f"{AZURE_STORAGE_ACCOUNT}"
)

print(
    f"✅ Storage Key carregada: "
    f"{bool(AZURE_STORAGE_KEY)}"
)

print(
    "✅ Inicialização concluída."
)

In [0]:
# =========================
# Configuração dos Clientes
# =========================

from google.cloud import bigquery
from azure.storage.blob import BlobServiceClient

# =========================
# Cliente BigQuery
# =========================

try:

    # ------------------------------------------
    # Validação credenciais GCP
    # ------------------------------------------

    if not cred_path:

        raise ValueError(
            "❌ Credenciais GCP não configuradas."
        )

    print(
        f"✅ Credenciais encontradas em: "
        f"{cred_path}"
    )

    # ------------------------------------------
    # Projeto Google Cloud
    # ------------------------------------------

    project_id = os.getenv(
        "GCP_PROJECT_ID",
        "tough-medley-505300-k1"
    )

    # ------------------------------------------
    # Inicializa cliente BigQuery
    # ------------------------------------------

    bq_client = (
        bigquery.Client
        .from_service_account_json(
            cred_path,
            project=project_id
        )
    )

    print(
        "✅ Cliente BigQuery inicializado com sucesso."
    )

except Exception as e:

    print(
        f"❌ Erro ao inicializar cliente BigQuery: {e}"
    )

    bq_client = None

# =========================
# Cliente Azure Blob
# =========================

try:

    # ------------------------------------------
    # Validações Azure
    # ------------------------------------------

    if not AZURE_STORAGE_ACCOUNT:

        raise ValueError(
            "❌ AZURE_STORAGE_ACCOUNT não configurado."
        )

    if not AZURE_STORAGE_KEY:

        raise ValueError(
            "❌ AZURE_STORAGE_KEY não configurado."
        )

    if not BRONZE_CONTAINER:

        raise ValueError(
            "❌ BRONZE_CONTAINER não configurado."
        )

    print(
        f"✅ Storage Account: "
        f"{AZURE_STORAGE_ACCOUNT}"
    )

    print(
        "✅ Storage Key carregada."
    )

    # ------------------------------------------
    # URL da Storage Account
    # ------------------------------------------

    account_url = (
        f"https://{AZURE_STORAGE_ACCOUNT}.blob.core.windows.net"
    )

    # ------------------------------------------
    # Inicializa Blob Storage
    # ------------------------------------------

    blob_service_client = BlobServiceClient(
        account_url=account_url,
        credential=AZURE_STORAGE_KEY
    )

    print(
        "✅ Cliente Azure Blob inicializado com sucesso."
    )

    # ------------------------------------------
    # Teste acesso ao container
    # ------------------------------------------

    container_client = (
        blob_service_client.get_container_client(
            BRONZE_CONTAINER
        )
    )

    container_client.get_container_properties()

    print(
        f"✅ Acesso validado ao container "
        f"'{BRONZE_CONTAINER}'."
    )

except Exception as e:

    print(
        f"❌ Erro ao inicializar cliente Azure Blob: {e}"
    )

    blob_service_client = None

# =========================
# Resumo
# =========================

print(
    f"✅ BigQuery disponível: "
    f"{bq_client is not None}"
)

print(
    f"✅ Azure Blob disponível: "
    f"{blob_service_client is not None}"
)

print(
    f"✅ Container Bronze: "
    f"{BRONZE_CONTAINER}"
)

print(
    "✅ Inicialização concluída."
)

In [0]:
# =========================
# Função de Exportação
# =========================

def export_bigquery_table_to_blob(
    source_table: str,
    blob_container: str,
    blob_name: str
):
    """
    Exporta uma tabela do BigQuery
    para Azure Blob Storage em formato Parquet.
    """

    try:

        # ------------------------------------------
        # Validação dos clientes
        # ------------------------------------------

        if bq_client is None:

            raise ValueError(
                "❌ Cliente BigQuery não inicializado."
            )

        if blob_service_client is None:

            raise ValueError(
                "❌ Cliente Azure Blob não inicializado."
            )

        if not blob_container:

            raise ValueError(
                "❌ Container Azure não informado."
            )

        if not blob_name:

            raise ValueError(
                "❌ Nome do blob não informado."
            )

        # ------------------------------------------
        # Consulta BigQuery
        # ------------------------------------------

        query = f"""
        SELECT *
        FROM `{source_table}`
        """

        print(
            f"✅ Executando consulta: "
            f"{source_table}"
        )

        query_job = bq_client.query(
            query,
            location="US"
        )

        # ------------------------------------------
        # DataFrame
        # ------------------------------------------

        df = query_job.to_dataframe()

        if df.empty:

            raise ValueError(
                f"❌ Nenhum registro retornado da tabela "
                f"{source_table}"
            )

        print(
            f"✅ Registros retornados: "
            f"{len(df):,}"
        )

        print(
            f"✅ Total de colunas: "
            f"{len(df.columns)}"
        )

        # ------------------------------------------
        # Auditoria
        # ------------------------------------------

        df["_ingested_at"] = (
            dt.datetime.now(
                dt.timezone.utc
            ).isoformat()
        )

        df["_source_table"] = (
            source_table
        )

        # ------------------------------------------
        # Diretório temporário
        # ------------------------------------------

        temp_dir = (
            BASE_DIR / "tmp"
        )

        temp_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        parquet_file = (
            temp_dir / blob_name
        )

        # ------------------------------------------
        # Salva parquet
        # ------------------------------------------

        df.to_parquet(
            parquet_file,
            index=False
        )

        if not parquet_file.exists():

            raise FileNotFoundError(
                f"❌ Arquivo parquet não criado: "
                f"{parquet_file}"
            )

        file_size_kb = round(
            parquet_file.stat().st_size / 1024,
            2
        )

        print(
            f"✅ Arquivo parquet criado: "
            f"{parquet_file}"
        )

        print(
            f"✅ Tamanho local: "
            f"{file_size_kb} KB"
        )

        # ------------------------------------------
        # Cliente Blob
        # ------------------------------------------

        blob_client = (
            blob_service_client.get_blob_client(
                container=blob_container,
                blob=blob_name
            )
        )

        # ------------------------------------------
        # Upload Azure Blob
        # ------------------------------------------

        with open(
            parquet_file,
            "rb"
        ) as file_data:

            blob_client.upload_blob(
                file_data,
                overwrite=True
            )

        # ------------------------------------------
        # Validação Upload
        # ------------------------------------------

        if not blob_client.exists():

            raise RuntimeError(
                f"❌ Upload não encontrado: "
                f"{blob_name}"
            )

        blob_properties = (
            blob_client.get_blob_properties()
        )

        # ------------------------------------------
        # Logs finais
        # ------------------------------------------

        print(
            f"✅ Exportação concluída."
        )

        print(
            f"✅ Tabela origem: "
            f"{source_table}"
        )

        print(
            f"✅ Container destino: "
            f"{blob_container}"
        )

        print(
            f"✅ Blob criado: "
            f"{blob_name}"
        )

        print(
            f"✅ Registros exportados: "
            f"{len(df):,}"
        )

        print(
            f"✅ Tamanho enviado: "
            f"{round(blob_properties.size / 1024, 2)} KB"
        )

        return {
            "success": True,
            "table": source_table,
            "container": blob_container,
            "blob": blob_name,
            "records": len(df)
        }

    except Exception as e:

        print(
            f"❌ Erro ao exportar "
            f"{source_table}: {e}"
        )

        raise

In [0]:
# =========================
# Ingestão Batch
# =========================

import datetime as dt

# ------------------------------------------
# Data de execução
# ------------------------------------------

execution_date = dt.datetime.now()

date_suffix = execution_date.strftime(
    "%Y-%m-%d"
)

print(
    f"✅ Data de execução: "
    f"{date_suffix}"
)

# ------------------------------------------
# Validação Container
# ------------------------------------------

if not BRONZE_CONTAINER:

    raise ValueError(
        "❌ Container Bronze não configurado."
    )

# ------------------------------------------
# Tabelas BigQuery
# ------------------------------------------

TABLES = [
    "basedosdados.br_inep_avaliacao_alfabetizacao.uf",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio"
    # "basedosdados.br_inep_avaliacao_alfabetizacao.municipio",
    # "basedosdados.br_inep_avaliacao_alfabetizacao.alunos"
]

# ------------------------------------------
# Validação
# ------------------------------------------

if not TABLES:

    raise ValueError(
        "❌ Nenhuma tabela configurada para ingestão."
    )

print(
    f"✅ Total de tabelas configuradas: "
    f"{len(TABLES)}"
)

# ------------------------------------------
# Loop de exportação
# ------------------------------------------

success_count = 0
error_count = 0

for table in TABLES:

    try:

        table_suffix = (
            table.split(".")[-1]
        )

        blob_name = (
            f"{date_suffix}_{table_suffix}.parquet"
        )

        print(
            "\n"
            "===================================="
        )

        print(
            f"▶ Processando tabela: "
            f"{table}"
        )

        export_bigquery_table_to_blob(
            source_table=table,
            blob_container=BRONZE_CONTAINER,
            blob_name=blob_name
        )

        success_count += 1

        print(
            f"✅ Exportação concluída: "
            f"{blob_name}"
        )

    except Exception as e:

        error_count += 1

        print(
            f"❌ Falha ao processar "
            f"{table}: {e}"
        )

# ------------------------------------------
# Resumo final
# ------------------------------------------

print(
    "\n"
    "===================================="
)

print(
    "✅ Ingestão Batch finalizada."
)

print(
    f"✅ Container destino: "
    f"{BRONZE_CONTAINER}"
)

print(
    f"✅ Tabelas processadas: "
    f"{len(TABLES)}"
)

print(
    f"✅ Sucesso: "
    f"{success_count}"
)

print(
    f"✅ Falhas: "
    f"{error_count}"
)

if error_count > 0:

    raise RuntimeError(
        f"❌ Processo finalizado com "
        f"{error_count} erro(s)."
    )

print(
    "✅ Todas as tabelas foram exportadas com sucesso."
)
print(
    "✅ Ingestão finalizada com sucesso."
)